In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv


In [2]:

import pandas as pd

# Load training and testing data
train = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv')
test = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv')

# Display first 5 rows
print(train.head())


   Id  MSSubClass MSZoning  LotFrontage  LotArea Street Alley LotShape  \
0   1          60       RL         65.0     8450   Pave   NaN      Reg   
1   2          20       RL         80.0     9600   Pave   NaN      Reg   
2   3          60       RL         68.0    11250   Pave   NaN      IR1   
3   4          70       RL         60.0     9550   Pave   NaN      IR1   
4   5          60       RL         84.0    14260   Pave   NaN      IR1   

  LandContour Utilities  ... PoolArea PoolQC Fence MiscFeature MiscVal MoSold  \
0         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      2   
1         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      5   
2         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      9   
3         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      2   
4         Lvl    AllPub  ...        0    NaN   NaN         NaN       0     12   

  YrSold  SaleType  SaleCondition  SalePrice  
0   2008        WD   

In [3]:

print("Training data shape:", train.shape)
print("Testing data shape:", test.shape)

print("\nColumn names:")
print(train.columns.tolist())


Training data shape: (1460, 81)
Testing data shape: (1459, 80)

Column names:
['Id', 'MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'MasVnrArea', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1', 'BsmtFinType2', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'Heating', 'HeatingQC', 'CentralAir', 'Electrical', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual', 'TotRmsAbvGrd', 'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType', 'GarageYrBlt', 'GarageFinish', 'GarageCars', 'GarageArea', 'GarageQual', 'GarageCond', 'PavedDrive', 'WoodDeckSF', 'O

In [4]:

# Count missing values in each column
missing = train.isnull().sum()

# Show columns that have missing values
print(missing[missing > 0].sort_values(ascending=False))


PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
BsmtExposure      38
BsmtFinType2      38
BsmtQual          37
BsmtCond          37
BsmtFinType1      37
MasVnrArea         8
Electrical         1
dtype: int64


In [5]:

# Check data types of columns
print(train.dtypes.value_counts())

# Check missing values in training and testing data
print("\nTraining missing values:", train.isnull().sum().sum())
print("Testing missing values:", test.isnull().sum().sum())


object     43
int64      35
float64     3
Name: count, dtype: int64

Training missing values: 7829
Testing missing values: 7878


In [6]:

# Separate target variable
y = train['SalePrice']

# Remove SalePrice and Id from training data
X = train.drop(['SalePrice', 'Id'], axis=1)

# Remove Id from testing data
X_test = test.drop(['Id'], axis=1)

print("Training features:", X.shape)
print("Target variable:", y.shape)
print("Testing features:", X_test.shape)


Training features: (1460, 79)
Target variable: (1460,)
Testing features: (1459, 79)


In [7]:

# Identify numerical columns
num_cols = X.select_dtypes(include=['int64', 'float64']).columns

# Identify categorical columns
cat_cols = X.select_dtypes(include=['object']).columns

print("Numerical columns:", len(num_cols))
print("Categorical columns:", len(cat_cols))


Numerical columns: 36
Categorical columns: 43


In [16]:

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor


In [17]:

# Process numerical columns
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

# Process categorical columns
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# Combine both pipelines
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

print("Preprocessing pipeline created successfully!")


Preprocessing pipeline created successfully!


In [18]:

# Create the complete ML pipeline
model = Pipeline([
    ('preprocessing', preprocessor),
    ('regressor', RandomForestRegressor(
        n_estimators=200,
        random_state=42
    ))
])

# Train the model
model.fit(X, y)

print("Model trained successfully!")


Model trained successfully!


In [19]:

# Predict prices for testing data
predictions = model.predict(X_test)

# Display first 10 predictions
print(predictions[:10])


[129354.54  155676.1   180121.31  181035.77  196502.6   184057.725
 169433.265 175817.965 182343.825 119307.955]


In [ ]:

# Create submission file
submission = pd.DataFrame({
    'Id': test['Id'],
    'SalePrice': predictions
})

# Save the file in Kaggle working folder
submission.to_csv('/kaggle/working/submission.csv', index=False)

# Display the first 5 rows
print(submission.head())

print("Submission file created successfully!")


In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_log_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
import numpy as np

# Split data into training and validation
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training rows:", X_train.shape[0])
print("Validation rows:", X_valid.shape[0])


In [4]:

import pandas as pd

print("Pandas imported successfully!")
print("Pandas version:", pd.__version__)


AttributeError: partially initialized module 'pandas' from '/usr/local/lib/python3.13/dist-packages/pandas/__init__.py' has no attribute 'plotting' (most likely due to a circular import)